# AGG + Neuropalsy — Results Analysis

This notebook loads the JSON output produced by `main.py` and generates:
- Summary statistics table
- Angular error distributions (healthy vs pathological)
- κ concentration histograms and 95%-cone plots
- Per-condition comparison across all evaluated disorders

**Pre-requisite:** run `python main.py` first to generate `gaze_results_*.json`.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

%matplotlib inline
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

## 1. Load results

In [ ]:
CONDITION = 'nystagmus'   # change as needed

with open(f'gaze_results_{CONDITION}.json') as f:
    data = json.load(f)

print('Condition :', data['condition'])
print('Severity  :', data['severity'])
print('κ clamp   :', data['kappa_clamp'])
print('SA restarts:', data['sa_restarts'], '× epochs:', data['sa_epochs'])

## 2. Summary statistics

In [ ]:
summary_df = pd.DataFrame([data['summary']])
summary_df.to_csv(f'summary_{CONDITION}.csv', index=False)
summary_df

## 3. Per-sample DataFrames

In [ ]:
df_healthy = pd.DataFrame(data['samples_healthy'])
df_patho   = pd.DataFrame(data['samples_pathological'])

df_healthy.to_csv(f'samples_healthy_{CONDITION}.csv',      index=False)
df_patho.to_csv(  f'samples_patho_{CONDITION}.csv',        index=False)

print(f'Healthy  samples : {len(df_healthy)}')
print(f'Patho    samples : {len(df_patho)}')
df_patho.head()

## 4. Angular error distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Healthy branch
ax = axes[0]
ax.hist(df_healthy['fc_err'],  bins=40, alpha=0.6, label='FC baseline',  color='steelblue')
ax.hist(df_healthy['gpm_err'], bins=40, alpha=0.6, label='GPM-AGG',      color='tomato')
ax.set_xlabel('Angular error (°)')
ax.set_ylabel('Count')
ax.set_title('Healthy gaze — angular error distribution')
ax.legend()

# Pathological branch
ax = axes[1]
ax.hist(df_patho['err_vs_clean'], bins=40, alpha=0.6, label='vMF μ vs clean GT', color='seagreen')
ax.hist(df_patho['err_vs_noisy'], bins=40, alpha=0.6, label='vMF μ vs noisy GT', color='orange')
ax.set_xlabel('Angular error (°)')
ax.set_title(f'Pathological gaze ({CONDITION}) — angular error distribution')
ax.legend()

plt.tight_layout()
plt.savefig(f'angular_error_{CONDITION}.pdf', bbox_inches='tight')
plt.show()

## 5. κ concentration & 95%-cone

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# κ histogram
ax = axes[0]
ax.hist(df_patho['kappa'], bins=40, color='mediumpurple', alpha=0.8)
ax.axvline(df_patho['kappa'].mean(), color='black', linestyle='--',
           label=f"mean κ = {df_patho['kappa'].mean():.2f}")
ax.set_xlabel('κ (concentration)')
ax.set_ylabel('Count')
ax.set_title('vMF concentration κ distribution')
ax.legend()

# 95%-cone
ax = axes[1]
ax.hist(df_patho['cone_95_deg'], bins=40, color='coral', alpha=0.8)
ax.axvline(df_patho['cone_95_deg'].mean(), color='black', linestyle='--',
           label=f"mean 95° cone = {df_patho['cone_95_deg'].mean():.1f}°")
ax.set_xlabel('95%-cone half-angle (°)')
ax.set_ylabel('Count')
ax.set_title('vMF 95% probability cone distribution')
ax.legend()

plt.tight_layout()
plt.savefig(f'kappa_cone_{CONDITION}.pdf', bbox_inches='tight')
plt.show()

## 6. Gaze direction helper

In [ ]:
import ast

def parse_vec(col):
    """Parse a column of '[x, y, z]' strings or lists into yaw/pitch (degrees)."""
    def to_angles(v):
        if isinstance(v, str):
            v = ast.literal_eval(v)
        x, y, z = v
        yaw   = np.degrees(np.arctan2(x, z))
        pitch = np.degrees(np.arcsin(np.clip(y, -1, 1)))
        return yaw, pitch
    return zip(*col.apply(to_angles))

df_patho['vmf_yaw'],   df_patho['vmf_pitch']   = parse_vec(df_patho['vmf_mu'])
df_patho['gt_yaw'],    df_patho['gt_pitch']     = parse_vec(df_patho['gt_noisy'])

df_patho[['vmf_yaw', 'vmf_pitch', 'gt_yaw', 'gt_pitch']].describe()

## 7. Scatter: predicted vs ground-truth gaze angles

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, comp, pred_col, gt_col, label in [
    (axes[0], 'Yaw',   'vmf_yaw',   'gt_yaw',   'Yaw (°)'),
    (axes[1], 'Pitch', 'vmf_pitch', 'gt_pitch',  'Pitch (°)'),
]:
    ax.scatter(df_patho[gt_col], df_patho[pred_col],
               alpha=0.15, s=4, color='steelblue')
    lim = max(abs(df_patho[gt_col]).max(), abs(df_patho[pred_col]).max()) + 5
    ax.plot([-lim, lim], [-lim, lim], 'r--', lw=1, label='ideal')
    ax.set_xlabel(f'GT {label}')
    ax.set_ylabel(f'Predicted {label}')
    ax.set_title(f'{comp}: vMF μ vs Ground Truth ({CONDITION})')
    ax.legend()

plt.tight_layout()
plt.savefig(f'scatter_{CONDITION}.pdf', bbox_inches='tight')
plt.show()